# Phase 3: Extract Python Test Cases from HuggingFace Dataset

This notebook loads the `newfacade/LeetCodeDataset` from HuggingFace, extracts Python test cases,
fixes indentation (adds 4 spaces before each `assert` line), and saves to `python_tests.jsonl`.

In [1]:
# Install required packages
import subprocess
import sys

# Install datasets if not already installed
subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "-q"])
print("✅ Dependencies installed")

✅ Dependencies installed


In [2]:
import json
from datasets import load_dataset
from pathlib import Path

print("Loading HuggingFace dataset: newfacade/LeetCodeDataset...")
dataset = load_dataset("newfacade/LeetCodeDataset")
print(f"✅ Dataset loaded: {dataset}")
print(f"\nDataset keys: {dataset.keys()}")

Loading HuggingFace dataset: newfacade/LeetCodeDataset...
✅ Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['task_id', 'question_id', 'difficulty', 'tags', 'problem_description', 'starter_code', 'estimated_date', 'prompt', 'completion', 'entry_point', 'test', 'input_output', 'query', 'response'],
        num_rows: 2641
    })
    test: Dataset({
        features: ['task_id', 'question_id', 'difficulty', 'tags', 'problem_description', 'starter_code', 'estimated_date', 'prompt', 'completion', 'entry_point', 'test', 'input_output', 'query', 'response'],
        num_rows: 228
    })
})

Dataset keys: dict_keys(['train', 'test'])


In [3]:
# Inspect first few samples to understand structure
if "train" in dataset:
    sample_split = dataset["train"]
else:
    sample_split = list(dataset.values())[0]

print(f"First sample keys: {sample_split[0].keys()}")
print(f"\nFirst sample:")
for key, value in sample_split[0].items():
    if isinstance(value, str) and len(value) > 100:
        print(f"  {key}: {value[:100]}...")
    else:
        print(f"  {key}: {value}")

First sample keys: dict_keys(['task_id', 'question_id', 'difficulty', 'tags', 'problem_description', 'starter_code', 'estimated_date', 'prompt', 'completion', 'entry_point', 'test', 'input_output', 'query', 'response'])

First sample:
  task_id: two-sum
  question_id: 1
  difficulty: Easy
  tags: ['Array', 'Hash Table']
  problem_description: Given an array of integers nums and an integer target, return indices of the two numbers such that t...
  starter_code: class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
        
  estimated_date: 2015-08-07 00:00:00
  prompt: import random
import functools
import collections
import string
import math
import datetime

from ty...
  completion: class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
        d = {}
    ...
  entry_point: Solution().twoSum
  test: def check(candidate):
    assert candidate(nums = [3, 3],target = 6) == [0, 1]
    assert candidate(...
  input_output: [{'input': 'nu

In [4]:
def fix_test_indentation(test_code: str) -> str:
    """
    Fix indentation in test code.
    Expected format:
        def check(candidate):
        assert candidate(...) == ...
        assert candidate(...) == ...
    
    Output format:
        def check(candidate):
            assert candidate(...) == ...
            assert candidate(...) == ...
    """
    lines = test_code.strip().split("\n")
    fixed_lines = []
    
    for line in lines:
        # First line should be 'def check(candidate):' - no indent
        if line.strip().startswith("def check"):
            fixed_lines.append(line.strip())
        # All other lines should have 4 spaces of indentation
        elif line.strip():
            # Remove existing indentation
            stripped = line.lstrip()
            # Add 4 spaces
            fixed_lines.append("    " + stripped)
    
    return "\n".join(fixed_lines)

# Test the function
test_input = """def check(candidate):
assert candidate(nums = [3, 3],target = 6) == [0, 1]
assert candidate(nums = [-1, -2, -3, -4],target = -8) == None"""

print("Before:")
print(repr(test_input))
print("\nAfter:")
print(repr(fix_test_indentation(test_input)))
print("\nFormatted:")
print(fix_test_indentation(test_input))

Before:
'def check(candidate):\nassert candidate(nums = [3, 3],target = 6) == [0, 1]\nassert candidate(nums = [-1, -2, -3, -4],target = -8) == None'

After:
'def check(candidate):\n    assert candidate(nums = [3, 3],target = 6) == [0, 1]\n    assert candidate(nums = [-1, -2, -3, -4],target = -8) == None'

Formatted:
def check(candidate):
    assert candidate(nums = [3, 3],target = 6) == [0, 1]
    assert candidate(nums = [-1, -2, -3, -4],target = -8) == None


In [5]:
# Extract test cases and save to JSONL
output_path = Path("/teamspace/studios/this_studio/dataset/complexity_reasoning_data/python_tests.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Determine which split to use
if "train" in dataset:
    data_split = dataset["train"]
else:
    data_split = list(dataset.values())[0]

print(f"Processing {len(data_split)} samples...")

extracted_count = 0
skipped_count = 0
error_count = 0

MAX_ASSERT_LINES = 30  # Keep only first 30 assert lines

with open(output_path, "w") as f:
    for i, sample in enumerate(data_split):
        try:
            # Try to find problem_id and test field
            problem_id = sample.get("question_id") or sample.get("problem_id") or sample.get("id")
            test_code = sample.get("test") or sample.get("test_code") or sample.get("tests")
            
            if problem_id and test_code:
                # Check if test contains 'def check' format
                if "def check" in str(test_code):
                    # Fix indentation
                    fixed_test = fix_test_indentation(test_code)
                    
                    # Limit to first 30 assert lines
                    lines = fixed_test.split("\n")
                    def_check_line = lines[0]  # "def check(candidate):"
                    assert_lines = [l for l in lines[1:] if l.strip().startswith("assert")]
                    limited_assert_lines = assert_lines[:MAX_ASSERT_LINES]
                    final_test = "\n".join([def_check_line] + limited_assert_lines)
                    
                    # Write to JSONL
                    json.dump({"problem_id": problem_id, "test": final_test}, f)
                    f.write("\n")
                    extracted_count += 1
                else:
                    skipped_count += 1
            else:
                skipped_count += 1
        except Exception as e:
            error_count += 1
            if error_count <= 5:  # Print first 5 errors
                print(f"Error at index {i}: {e}")
        
        if (i + 1) % 500 == 0:
            print(f"  Processed {i + 1}/{len(data_split)} samples... (extracted: {extracted_count}, skipped: {skipped_count})")

print(f"\n✅ Extraction complete!")
print(f"  Extracted: {extracted_count} test cases")
print(f"  Skipped: {skipped_count}")
print(f"  Errors: {error_count}")
print(f"  Output: {output_path}")

Processing 2641 samples...


  Processed 500/2641 samples... (extracted: 500, skipped: 0)
  Processed 1000/2641 samples... (extracted: 1000, skipped: 0)
  Processed 1500/2641 samples... (extracted: 1500, skipped: 0)
  Processed 2000/2641 samples... (extracted: 2000, skipped: 0)
  Processed 2500/2641 samples... (extracted: 2500, skipped: 0)

✅ Extraction complete!
  Extracted: 2641 test cases
  Skipped: 0
  Errors: 0
  Output: /teamspace/studios/this_studio/dataset/complexity_reasoning_data/python_tests.jsonl


In [6]:
# Verify output and show samples
import os

file_size = os.path.getsize(output_path)
line_count = sum(1 for _ in open(output_path))

print(f"Output file: {output_path}")
print(f"  File size: {file_size:,} bytes")
print(f"  Line count: {line_count}")
print(f"\nFirst 3 samples:")

with open(output_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        sample = json.loads(line)
        print(f"\n  Sample {i+1}:")
        print(f"    problem_id: {sample['problem_id']}")
        test_lines = sample['test'].split("\n")
        print(f"    test: (first 3 lines)")
        for j, tline in enumerate(test_lines[:3]):
            print(f"      {repr(tline)}")
        if len(test_lines) > 3:
            print(f"      ... ({len(test_lines) - 3} more lines)")

Output file: /teamspace/studios/this_studio/dataset/complexity_reasoning_data/python_tests.jsonl
  File size: 10,779,921 bytes
  Line count: 2641

First 3 samples:

  Sample 1:
    problem_id: 1
    test: (first 3 lines)
      'def check(candidate):'
      '    assert candidate(nums = [3, 3],target = 6) == [0, 1]'
      '    assert candidate(nums = [-1, -2, -3, -4],target = -8) == None'
      ... (28 more lines)

  Sample 2:
    problem_id: 2
    test: (first 3 lines)
      'def check(candidate):'
      '    assert is_same_list(candidate(l1 = list_node([9, 8, 7]),l2 = list_node([1, 2, 3])), list_node([0, 1, 1, 1]))'
      '    assert is_same_list(candidate(l1 = list_node([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),l2 = list_node([5, 6, 4])), list_node([6, 6, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))'
      ... (28 more lines)

  Sample 3:
    problem_id: 3
    test: (first 3 lines)
      '